# YOLO11m Football Detector Training

## Before running in Kaggle

1. Turn **Internet** on.
2. Set **Accelerator** to **GPU T4 x2** if available. The notebook auto-detects GPUs.
3. Add your Roboflow YOLO dataset as a Kaggle input.
4. Run cells top-to-bottom.

## What this version does

- Uses **YOLO11m** (`yolo11m.pt`).
- Keeps resume-safe checkpoints with `last.pt` and `save_period=1`.
- Auto-refreshes `/kaggle/working/MUST_DOWNLOAD_yolo11m_weights.zip` after each epoch.
- Adds football-specific augmentations:
  - random-angle linear motion blur from 0° to 360°
  - random scaling from 0.5x to 1.5x plus random crop/pad
  - HSV, brightness, and contrast shifts
  - YOLO mosaic on, MixUp off/near-zero


In [ ]:
# =========================
# 0. INSTALL PACKAGES
# =========================

!pip -q install --no-deps ultralytics ultralytics-thop

In [ ]:
# =========================
# 1. IMPORTS + GPU CHECK
# =========================

import os
import glob
import shutil
import zipfile
import random
from pathlib import Path

import cv2
import yaml
import numpy as np
from tqdm import tqdm
import torch
from ultralytics import YOLO

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU 0:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU found. In Kaggle settings, set Accelerator to GPU T4 x2, then restart session.")

In [ ]:
# =========================
# 2. CONFIG
# =========================

# Core training settings
MODEL_NAME = "yolo11m.pt"
RUN_NAME = "football_detector_yolo11m_aug"

EPOCHS = 75
IMAGE_SIZE = 960

# Multi-GPU is faster on Kaggle T4 x2. If Kaggle/Ultralytics DDP gives you trouble, set this False.
USE_MULTI_GPU_IF_AVAILABLE = False

# Batch:
# - Multi-GPU: fixed batch is safer.
# - Single-GPU: batch=-1 lets Ultralytics auto-pick ~60% GPU memory use.
BATCH_SIZE_SINGLE_GPU = -1
BATCH_SIZE_MULTI_GPU = -1

PATIENCE = 25
WORKERS = 8
CACHE = "disk"       # "disk" is usually safer than RAM on Kaggle
SAVE_PERIOD = 1      # save every epoch so last.pt can resume after pauses

# Resume behavior
AUTO_RESUME = True
ALLOW_RESUME_FROM_INPUT_ZIP = True
FORCE_RESTART = False  # set True only when you intentionally want to delete the old YOLO11m run

# Dataset settings
VAL_FRACTION = 0.15
RANDOM_SEED = 42

# ======================================================================
# FOOTBALL-SPECIFIC OFFLINE AUGMENTATIONS
# These create augmented training images before YOLO training starts.
# YOLO's own online augmentations are also configured in the train cell.
# ======================================================================
AUGMENTED_COPIES_PER_IMAGE = 1  # set to 2 if you want heavier offline augmentation, but training will be slower

# 1) High-speed motion blur
USE_RANDOM_MOTION_BLUR = True
MOTION_BLUR_PROB = 0.75
MOTION_BLUR_LENGTH_MIN = 15      # pixels
MOTION_BLUR_LENGTH_MAX = 100     # pixels
MOTION_BLUR_ANGLE_MIN = 0.0      # degrees
MOTION_BLUR_ANGLE_MAX = 360.0    # degrees

# 2) Resolution simulation / scaling / random crop
USE_RANDOM_SCALE_CROP = True
SCALE_CROP_PROB = 1.0
SCALE_MIN = 0.50                 # 0.5x zoom-out/downsample simulation
SCALE_MAX = 1.50                 # 1.5x zoom-in/crop simulation
MIN_BOX_AREA_KEPT = 0.10         # drop box if crop leaves <10% of its transformed area

# Existing useful spatial augmentation
USE_HORIZONTAL_FLIP = True
HORIZONTAL_FLIP_PROB = 0.5

# 4) Background / lighting robustness
USE_RANDOM_HSV = True
HUE_SHIFT_DEGREES = 10           # random hue shift between -10 and +10 OpenCV hue degrees
SATURATION_MIN = 0.70            # -30%
SATURATION_MAX = 1.30            # +30%
VALUE_MIN = 0.80                 # darker stadium / shadow
VALUE_MAX = 1.20                 # brighter stadium lights / noon sun

USE_RANDOM_BRIGHTNESS_CONTRAST = True
BRIGHTNESS_DELTA = 30            # random additive brightness shift in [-30, +30]
CONTRAST_MIN = 0.80
CONTRAST_MAX = 1.25

# Built-in YOLO augmentation settings for small footballs
YOLO_HSV_H = 0.03
YOLO_HSV_S = 0.35
YOLO_HSV_V = 0.25
YOLO_TRANSLATE = 0.10
YOLO_SCALE = 0.50                # final scale range is 0.5x to 1.5x
YOLO_MULTI_SCALE = 0.50          # batch image-size variation, approx 0.5x to 1.5x
YOLO_MOSAIC = 1.00               # strong small-object help
YOLO_MIXUP = 0.00                # off so tiny footballs do not disappear into blended textures
YOLO_CLOSE_MOSAIC = 10           # turn mosaic off near the end for final stabilization

# Working dirs
RAW_DIR = Path("/kaggle/working/raw_dataset")
AUG_DIR = Path("/kaggle/working/aug_dataset")
RUNS_DIR = Path("/kaggle/working/runs")
RUN_DIR = RUNS_DIR / RUN_NAME

ZIP_PATH = Path("/kaggle/working/MUST_DOWNLOAD_yolo11m_weights.zip")
BEST_COPY_PATH = Path("/kaggle/working/best_yolo11m.pt")
LAST_COPY_PATH = Path("/kaggle/working/last_yolo11m.pt")

IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]


In [ ]:
# =========================
# 3. FIND / COPY DATASET
# =========================

# Fresh start for the dataset copy/augmentation only.
# This does NOT delete /kaggle/working/runs, so training checkpoints can still resume.
for d in [RAW_DIR, AUG_DIR]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

def zip_looks_like_yolo_dataset(zip_path):
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            names = zf.namelist()

            # Skip weights/resume zips created by this notebook.
            has_resume_metadata = any(n.endswith("MODEL_NAME.txt") for n in names) and any(n.endswith("RUN_NAME.txt") for n in names)
            if has_resume_metadata:
                return False

            # Roboflow YOLO exports should contain data.yaml.
            return any(Path(n).name == "data.yaml" for n in names)

    except zipfile.BadZipFile:
        return False

# Look for a dataset zip first, but ignore checkpoint/download zips.
all_zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
dataset_zips = [z for z in all_zips if zip_looks_like_yolo_dataset(z)]

if len(dataset_zips) > 0:
    zip_path = dataset_zips[0]
    print("Found dataset zip:", zip_path)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(RAW_DIR)

else:
    print("No dataset zip found. Looking for data.yaml folder in /kaggle/input...")
    yamls = glob.glob("/kaggle/input/**/data.yaml", recursive=True)

    if len(yamls) == 0:
        raise FileNotFoundError("Could not find a dataset zip or data.yaml. Attach your Roboflow YOLO dataset to the notebook.")

    input_root = Path(yamls[0]).parent
    print("Found dataset folder:", input_root)

    # Copy read-only Kaggle input dataset into working dir
    shutil.copytree(input_root, RAW_DIR, dirs_exist_ok=True)

# Find data.yaml
yaml_files = glob.glob(str(RAW_DIR / "**" / "data.yaml"), recursive=True)
if len(yaml_files) == 0:
    raise FileNotFoundError("Could not find data.yaml after extracting/copying dataset.")

DATA_YAML = Path(yaml_files[0])
DATASET_ROOT = DATA_YAML.parent

print("Dataset root:", DATASET_ROOT)
print("data.yaml:", DATA_YAML)

# Copy to augmented working dataset
shutil.copytree(DATASET_ROOT, AUG_DIR, dirs_exist_ok=True)
print("Copied dataset to:", AUG_DIR)


In [ ]:
# =========================
# 4. NORMALIZE FOLDER NAMES + CREATE VALIDATION IF MISSING
# =========================

def get_images(folder):
    paths = []
    if not folder.exists():
        return []
    for ext in IMG_EXTS:
        paths.extend(folder.glob(f"*{ext}"))
        paths.extend(folder.glob(f"*{ext.upper()}"))
    return sorted(list(set(paths)))

# Some datasets use val/ instead of valid/.
if (AUG_DIR / "val").exists() and not (AUG_DIR / "valid").exists():
    shutil.move(str(AUG_DIR / "val"), str(AUG_DIR / "valid"))

train_img_dir = AUG_DIR / "train" / "images"
train_label_dir = AUG_DIR / "train" / "labels"
valid_img_dir = AUG_DIR / "valid" / "images"
valid_label_dir = AUG_DIR / "valid" / "labels"

if not train_img_dir.exists():
    raise FileNotFoundError(f"Could not find train images folder: {train_img_dir}")

if not train_label_dir.exists():
    raise FileNotFoundError(f"Could not find train labels folder: {train_label_dir}")

# If no validation set exists, split 15% from train
if not valid_img_dir.exists():
    print("No valid/images folder found. Creating validation split from train...")

    valid_img_dir.mkdir(parents=True, exist_ok=True)
    valid_label_dir.mkdir(parents=True, exist_ok=True)

    original_train_images = get_images(train_img_dir)

    random.seed(RANDOM_SEED)
    random.shuffle(original_train_images)

    num_val = max(1, int(len(original_train_images) * VAL_FRACTION))
    val_images = original_train_images[:num_val]

    print("Moving to validation:", len(val_images))

    for img_path in val_images:
        label_path = train_label_dir / f"{img_path.stem}.txt"

        shutil.move(str(img_path), str(valid_img_dir / img_path.name))

        if label_path.exists():
            shutil.move(str(label_path), str(valid_label_dir / label_path.name))

else:
    print("Validation set already exists. Not creating a new split.")

print("After validation split:")
print("Train images:", len(get_images(train_img_dir)))
print("Valid images:", len(get_images(valid_img_dir)))

In [ ]:
# =========================
# 5. YOLO LABEL HELPERS
# =========================

def read_yolo_label(label_path):
    boxes = []

    if not label_path.exists():
        return boxes

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            cls, x, y, w, h = parts
            boxes.append([int(float(cls)), float(x), float(y), float(w), float(h)])

    return boxes


def write_yolo_label(label_path, boxes):
    with open(label_path, "w") as f:
        for cls, x, y, w, h in boxes:
            x = min(max(x, 0.0), 1.0)
            y = min(max(y, 0.0), 1.0)
            w = min(max(w, 0.0), 1.0)
            h = min(max(h, 0.0), 1.0)

            if w <= 0 or h <= 0:
                continue

            f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

In [ ]:
# =========================
# 6. CUSTOM AUGMENTATION FUNCTIONS
# =========================


def horizontal_flip(image, boxes):
    flipped = cv2.flip(image, 1)

    new_boxes = []
    for cls, x, y, w, h in boxes:
        new_x = 1.0 - x
        new_boxes.append([cls, new_x, y, w, h])

    return flipped, new_boxes


def yolo_to_xyxy(box, img_w, img_h):
    cls, x, y, w, h = box
    x1 = (x - w / 2) * img_w
    y1 = (y - h / 2) * img_h
    x2 = (x + w / 2) * img_w
    y2 = (y + h / 2) * img_h
    return cls, x1, y1, x2, y2


def xyxy_to_yolo(cls, x1, y1, x2, y2, img_w, img_h):
    x1 = np.clip(x1, 0, img_w)
    y1 = np.clip(y1, 0, img_h)
    x2 = np.clip(x2, 0, img_w)
    y2 = np.clip(y2, 0, img_h)

    bw = x2 - x1
    bh = y2 - y1
    if bw <= 1 or bh <= 1:
        return None

    x = (x1 + x2) / 2 / img_w
    y = (y1 + y2) / 2 / img_h
    w = bw / img_w
    h = bh / img_h
    return [cls, float(x), float(y), float(w), float(h)]


def random_scale_crop_pad(image, boxes, scale_factor=None):
    """Randomly scales image 0.5x-1.5x, then crops or pads back to original size.

    This simulates broadcast zoom levels and All-22 wide shots while preserving YOLO labels.
    """
    orig_h, orig_w = image.shape[:2]
    if scale_factor is None:
        scale_factor = np.random.uniform(SCALE_MIN, SCALE_MAX)

    new_w = max(2, int(round(orig_w * scale_factor)))
    new_h = max(2, int(round(orig_h * scale_factor)))

    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    # Convert boxes to scaled pixel coordinates.
    scaled_boxes = []
    for box in boxes:
        cls, x1, y1, x2, y2 = yolo_to_xyxy(box, orig_w, orig_h)
        scaled_boxes.append([cls, x1 * scale_factor, y1 * scale_factor, x2 * scale_factor, y2 * scale_factor])

    if scale_factor >= 1.0:
        # Zoom-in case: random crop back to original resolution.
        max_x0 = max(0, new_w - orig_w)
        max_y0 = max(0, new_h - orig_h)
        x0 = np.random.randint(0, max_x0 + 1) if max_x0 > 0 else 0
        y0 = np.random.randint(0, max_y0 + 1) if max_y0 > 0 else 0

        out = resized[y0:y0 + orig_h, x0:x0 + orig_w]

        # If rounding made the crop slightly small, pad the edge.
        if out.shape[0] != orig_h or out.shape[1] != orig_w:
            fixed = np.zeros((orig_h, orig_w, 3), dtype=image.dtype)
            fixed[:out.shape[0], :out.shape[1]] = out
            out = fixed

        dx, dy = -x0, -y0

    else:
        # Zoom-out/downsample case: place resized image on a canvas at a random offset.
        out = np.zeros_like(image)
        max_x0 = max(0, orig_w - new_w)
        max_y0 = max(0, orig_h - new_h)
        x0 = np.random.randint(0, max_x0 + 1) if max_x0 > 0 else 0
        y0 = np.random.randint(0, max_y0 + 1) if max_y0 > 0 else 0
        out[y0:y0 + new_h, x0:x0 + new_w] = resized
        dx, dy = x0, y0

    new_boxes = []
    for cls, x1, y1, x2, y2 in scaled_boxes:
        tx1, ty1, tx2, ty2 = x1 + dx, y1 + dy, x2 + dx, y2 + dy

        before_area = max(0.0, tx2 - tx1) * max(0.0, ty2 - ty1)
        clipped_x1 = np.clip(tx1, 0, orig_w)
        clipped_y1 = np.clip(ty1, 0, orig_h)
        clipped_x2 = np.clip(tx2, 0, orig_w)
        clipped_y2 = np.clip(ty2, 0, orig_h)
        after_area = max(0.0, clipped_x2 - clipped_x1) * max(0.0, clipped_y2 - clipped_y1)

        if before_area <= 0 or after_area / before_area < MIN_BOX_AREA_KEPT:
            continue

        yolo_box = xyxy_to_yolo(cls, clipped_x1, clipped_y1, clipped_x2, clipped_y2, orig_w, orig_h)
        if yolo_box is not None:
            new_boxes.append(yolo_box)

    return out, new_boxes


def random_hsv_bgr(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV).astype(np.float32)

    # OpenCV hue is stored in [0, 179]. Rotate hue with wrap-around.
    hue_delta = np.random.uniform(-HUE_SHIFT_DEGREES, HUE_SHIFT_DEGREES)
    sat_factor = np.random.uniform(SATURATION_MIN, SATURATION_MAX)
    val_factor = np.random.uniform(VALUE_MIN, VALUE_MAX)

    hsv[:, :, 0] = (hsv[:, :, 0] + hue_delta) % 180
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * sat_factor, 0, 255)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * val_factor, 0, 255)

    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)


def random_brightness_contrast(image):
    contrast = np.random.uniform(CONTRAST_MIN, CONTRAST_MAX)
    brightness = np.random.uniform(-BRIGHTNESS_DELTA, BRIGHTNESS_DELTA)
    out = image.astype(np.float32) * contrast + brightness
    return np.clip(out, 0, 255).astype(np.uint8)


def linear_motion_blur(image, length=None, angle_degrees=None):
    """Apply linear motion blur at an arbitrary angle from 0 to 360 degrees."""
    if length is None:
        length = np.random.randint(MOTION_BLUR_LENGTH_MIN, MOTION_BLUR_LENGTH_MAX + 1)
    if angle_degrees is None:
        angle_degrees = np.random.uniform(MOTION_BLUR_ANGLE_MIN, MOTION_BLUR_ANGLE_MAX)

    length = int(max(3, length))
    if length % 2 == 0:
        length += 1

    kernel = np.zeros((length, length), dtype=np.float32)
    kernel[length // 2, :] = 1.0

    center = (length // 2, length // 2)
    rotation = cv2.getRotationMatrix2D(center, angle_degrees, 1.0)
    kernel = cv2.warpAffine(kernel, rotation, (length, length))

    kernel_sum = kernel.sum()
    if kernel_sum <= 0:
        return image

    kernel /= kernel_sum
    return cv2.filter2D(image, -1, kernel, borderType=cv2.BORDER_REPLICATE)


def augment_one_image(image, boxes):
    # Random scale 0.5x-1.5x + crop/pad back to original size.
    if USE_RANDOM_SCALE_CROP and np.random.rand() < SCALE_CROP_PROB:
        image, boxes = random_scale_crop_pad(image, boxes)

    # Horizontal flip.
    if USE_HORIZONTAL_FLIP and np.random.rand() < HORIZONTAL_FLIP_PROB:
        image, boxes = horizontal_flip(image, boxes)

    # HSV + value shifts for turf / lighting / jersey-color robustness.
    if USE_RANDOM_HSV:
        image = random_hsv_bgr(image)

    # Brightness + contrast for noon sun, shadows, stadium lights, rain/snow broadcast exposure.
    if USE_RANDOM_BRIGHTNESS_CONTRAST:
        image = random_brightness_contrast(image)

    # Random-angle linear motion blur from 0 to 360 degrees.
    if USE_RANDOM_MOTION_BLUR and np.random.rand() < MOTION_BLUR_PROB:
        image = linear_motion_blur(image)

    return image, boxes


In [ ]:
# =========================
# 7. APPLY AUGMENTATIONS TO TRAIN ONLY
# =========================

train_images_before = get_images(train_img_dir)

# Only augment non-augmented originals.
train_images_to_augment = [p for p in train_images_before if "_aug" not in p.stem]

print("Training originals to augment:", len(train_images_to_augment))
print("Augmented copies per image:", AUGMENTED_COPIES_PER_IMAGE)
print("\nAugmentation checklist:")
print(f"1. Random-angle linear motion blur: {USE_RANDOM_MOTION_BLUR}, p={MOTION_BLUR_PROB}, length={MOTION_BLUR_LENGTH_MIN}-{MOTION_BLUR_LENGTH_MAX}px, angle={MOTION_BLUR_ANGLE_MIN}-{MOTION_BLUR_ANGLE_MAX} deg")
print(f"2. Random scale/crop: {USE_RANDOM_SCALE_CROP}, p={SCALE_CROP_PROB}, scale={SCALE_MIN}-{SCALE_MAX}x")
print(f"3. YOLO Mosaic/MixUp configured in train cell: mosaic={YOLO_MOSAIC}, mixup={YOLO_MIXUP}")
print(f"4. HSV + brightness/contrast: HSV={USE_RANDOM_HSV}, brightness/contrast={USE_RANDOM_BRIGHTNESS_CONTRAST}")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

created = 0
for img_path in tqdm(train_images_to_augment):
    image = cv2.imread(str(img_path))

    if image is None:
        print("Skipping unreadable image:", img_path)
        continue

    label_path = train_label_dir / f"{img_path.stem}.txt"
    boxes = read_yolo_label(label_path)

    for copy_idx in range(1, AUGMENTED_COPIES_PER_IMAGE + 1):
        aug_image, aug_boxes = augment_one_image(image.copy(), [b[:] for b in boxes])

        aug_img_path = train_img_dir / f"{img_path.stem}_aug{copy_idx}{img_path.suffix}"
        aug_label_path = train_label_dir / f"{img_path.stem}_aug{copy_idx}.txt"

        cv2.imwrite(str(aug_img_path), aug_image)
        write_yolo_label(aug_label_path, aug_boxes)
        created += 1

print("After augmentation:")
print("Created augmented images:", created)
print("Train images:", len(get_images(train_img_dir)))
print("Valid images:", len(get_images(valid_img_dir)))


In [ ]:
# =========================
# 8. FIX data.yaml ABSOLUTE PATHS
# =========================

aug_yaml_path = AUG_DIR / "data.yaml"

with open(aug_yaml_path, "r") as f:
    data = yaml.safe_load(f)

data["train"] = str(AUG_DIR / "train" / "images")
data["val"] = str(AUG_DIR / "valid" / "images")

if (AUG_DIR / "test" / "images").exists():
    data["test"] = str(AUG_DIR / "test" / "images")

with open(aug_yaml_path, "w") as f:
    yaml.safe_dump(data, f)

print("Final data.yaml:")
print(open(aug_yaml_path).read())

In [ ]:
# =========================
# 9. DELETE OLD YOLO CACHE FILES + SANITY CHECK
# =========================

cache_files = [
    AUG_DIR / "train" / "labels.cache",
    AUG_DIR / "valid" / "labels.cache",
    AUG_DIR / "test" / "labels.cache",
]

for cache_file in cache_files:
    if cache_file.exists():
        cache_file.unlink()
        print("Deleted cache:", cache_file)

train_img_count = len(get_images(AUG_DIR / "train" / "images"))
train_label_count = len(list((AUG_DIR / "train" / "labels").glob("*.txt")))
valid_img_count = len(get_images(AUG_DIR / "valid" / "images"))
valid_label_count = len(list((AUG_DIR / "valid" / "labels").glob("*.txt")))

print("========== FINAL DATASET COUNTS ==========")
print("Train images:", train_img_count)
print("Train labels:", train_label_count)
print("Valid images:", valid_img_count)
print("Valid labels:", valid_label_count)
print("==========================================")

if train_img_count == 0 or valid_img_count == 0:
    raise RuntimeError("Train or validation set is empty.")

print("Expected for your previous dataset was about: Train images 2424, Valid images 213")

In [ ]:
from pathlib import Path

last = Path("/kaggle/working/runs/football_detector_yolo11m_aug/weights/last.pt")
best = Path("/kaggle/working/runs/football_detector_yolo11m_aug/weights/best.pt")

print("last.pt exists:", last.exists())
print("best.pt exists:", best.exists())

In [ ]:
# =========================
# RESTORE UPLOADED last.pt / best.pt FROM KAGGLE INPUT
# =========================

from pathlib import Path
import shutil

WORK_DIR = Path("/kaggle/working")
RUN_NAME = "football_detector_yolo11m_aug"
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
WEIGHTS_DIR = RUN_DIR / "weights"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# Find uploaded weights in Kaggle inputs
input_last = list(Path("/kaggle/input").rglob("last.pt"))
input_best = list(Path("/kaggle/input").rglob("best.pt"))

print("Found uploaded last.pt:", input_last)
print("Found uploaded best.pt:", input_best)

if not input_last:
    print("No uploaded last.pt found. Training will start from yolo11m.pt unless local last.pt exists.")
else:
    shutil.copy2(input_last[0], WEIGHTS_DIR / "last.pt")
    print("Restored last.pt to:", WEIGHTS_DIR / "last.pt")

if input_best:
    shutil.copy2(input_best[0], WEIGHTS_DIR / "best.pt")
    print("Restored best.pt to:", WEIGHTS_DIR / "best.pt")

print("Ready to resume:", (WEIGHTS_DIR / "last.pt").exists())

In [ ]:
# =========================
# 10. TRAIN YOLO11m WITH SAFE AUTO-RESUME
# =========================

from pathlib import Path
import os
import shutil
import zipfile
import torch
from IPython.display import FileLink, display, Javascript
from ultralytics import YOLO

def choose_device_and_batch():
    gpu_count = torch.cuda.device_count()

    if gpu_count <= 0:
        raise RuntimeError("No GPU found. In Kaggle settings, set Accelerator to GPU, then restart session.")

    if gpu_count >= 2 and USE_MULTI_GPU_IF_AVAILABLE:
        device = list(range(gpu_count))   # e.g. [0, 1] on Kaggle T4 x2
        batch = BATCH_SIZE_MULTI_GPU
    else:
        device = 0
        batch = BATCH_SIZE_SINGLE_GPU

    return device, batch


def zip_current_artifacts(zip_path=ZIP_PATH, quiet=False):
    # Creates/refreshes MUST_DOWNLOAD_yolo11m_weights.zip.
    # Safe to run even if training did not finish yet, as long as last.pt exists.
    weights_dir = RUN_DIR / "weights"
    best_pt = weights_dir / "best.pt"
    last_pt = weights_dir / "last.pt"

    if not best_pt.exists() and not last_pt.exists():
        if not quiet:
            print("No best.pt or last.pt found yet. Finish at least 1 epoch first.")
        return None

    # Copy the important files to the top level of /kaggle/working for easy download too.
    if best_pt.exists():
        shutil.copy2(best_pt, BEST_COPY_PATH)
    if last_pt.exists():
        shutil.copy2(last_pt, LAST_COPY_PATH)

    metadata_dir = Path("/kaggle/working/yolo11m_export_metadata")
    metadata_dir.mkdir(parents=True, exist_ok=True)
    (metadata_dir / "MODEL_NAME.txt").write_text(MODEL_NAME)
    (metadata_dir / "RUN_NAME.txt").write_text(RUN_NAME)
    (metadata_dir / "IMAGE_SIZE.txt").write_text(str(IMAGE_SIZE))
    (metadata_dir / "EPOCHS_TARGET.txt").write_text(str(EPOCHS))
    (metadata_dir / "AUGMENTATION_SETTINGS.txt").write_text(
        "motion_blur=random_angle_0_360; "
        f"motion_blur_prob={MOTION_BLUR_PROB}; "
        f"motion_blur_length={MOTION_BLUR_LENGTH_MIN}-{MOTION_BLUR_LENGTH_MAX}; "
        f"scale_crop={SCALE_MIN}-{SCALE_MAX}; "
        f"yolo_mosaic={YOLO_MOSAIC}; yolo_mixup={YOLO_MIXUP}; "
        f"yolo_scale={YOLO_SCALE}; yolo_multi_scale={YOLO_MULTI_SCALE}; "
        f"hsv_h={YOLO_HSV_H}; hsv_s={YOLO_HSV_S}; hsv_v={YOLO_HSV_V}"
    )

    include_files = []

    # Weights/checkpoints
    include_files.extend(list(weights_dir.glob("*.pt")))

    # Useful training artifacts
    for pattern in [
        "args.yaml",
        "results.csv",
        "results.png",
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
        "F1_curve.png",
        "P_curve.png",
        "PR_curve.png",
        "R_curve.png",
        "labels.jpg",
        "train_batch*.jpg",
        "val_batch*.jpg",
    ]:
        include_files.extend(list(RUN_DIR.glob(pattern)))

    # Dataset yaml used for training
    if "aug_yaml_path" in globals() and Path(aug_yaml_path).exists():
        include_files.append(Path(aug_yaml_path))

    include_files.extend(list(metadata_dir.glob("*.txt")))

    # Deduplicate while preserving order
    seen = set()
    unique_files = []
    for p in include_files:
        p = Path(p)
        if p.exists() and p not in seen:
            unique_files.append(p)
            seen.add(p)

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in unique_files:
            if p.is_relative_to(Path("/kaggle/working")):
                arcname = p.relative_to(Path("/kaggle/working"))
            else:
                arcname = p.name
            zf.write(p, arcname=str(arcname))

    if not quiet:
        print(f"Created/updated: {zip_path}")
        print(f"Zip size: {zip_path.stat().st_size / (1024*1024):.1f} MB")

    return zip_path


def on_epoch_end_make_zip(trainer):
    # Auto-refresh zip after every epoch, so if Kaggle pauses later you still have a recent downloadable checkpoint.
    try:
        zip_current_artifacts(quiet=True)
    except Exception as e:
        print("Checkpoint zip refresh failed, but training will continue:", repr(e))


def find_compatible_input_resume_zip():
    # Looks for a previous MUST_DOWNLOAD_yolo11m_weights.zip added as a Kaggle input.
    # To avoid accidentally resuming YOLO12/other runs, it only accepts zips with matching metadata.
    if not ALLOW_RESUME_FROM_INPUT_ZIP:
        return None

    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None

    resume_extract_dir = Path("/kaggle/working/resume_from_input_zip")
    resume_extract_dir.mkdir(parents=True, exist_ok=True)

    for zip_path in sorted(input_root.glob("**/*.zip")):
        try:
            with zipfile.ZipFile(zip_path, "r") as zf:
                names = zf.namelist()

                model_meta = [n for n in names if n.endswith("MODEL_NAME.txt")]
                run_meta = [n for n in names if n.endswith("RUN_NAME.txt")]

                if not model_meta or not run_meta:
                    continue

                model_name = zf.read(model_meta[0]).decode("utf-8", errors="ignore").strip()
                run_name = zf.read(run_meta[0]).decode("utf-8", errors="ignore").strip()

                if model_name != MODEL_NAME or run_name != RUN_NAME:
                    continue

                last_candidates = [n for n in names if n.endswith("last.pt")]
                if not last_candidates:
                    continue

                # Prefer last.pt over best.pt for resume because it stores the optimizer/scheduler state.
                member = sorted(last_candidates, key=len)[0]
                zf.extract(member, resume_extract_dir)
                extracted = resume_extract_dir / member
                print("Found compatible resume checkpoint inside input zip:", zip_path)
                print("Extracted:", extracted)
                return extracted

        except zipfile.BadZipFile:
            continue

    return None


# Optional true fresh start
if FORCE_RESTART and RUN_DIR.exists():
    print("FORCE_RESTART=True, deleting old run:", RUN_DIR)
    shutil.rmtree(RUN_DIR)

DEVICE, BATCH_SIZE = choose_device_and_batch()
print("Using device:", DEVICE)
print("Using batch:", BATCH_SIZE)

local_last = RUN_DIR / "weights" / "last.pt"
input_last = find_compatible_input_resume_zip()

# Resume only from matching YOLO11m run/checkpoint.
resume_ckpt = None
if AUTO_RESUME and local_last.exists():
    resume_ckpt = local_last
elif AUTO_RESUME and input_last is not None:
    resume_ckpt = input_last

if resume_ckpt is not None:
    print("Resuming interrupted YOLO11m training from:")
    print(resume_ckpt)

    model = YOLO(str(resume_ckpt))
    model.add_callback("on_train_epoch_end", on_epoch_end_make_zip)
    results = model.train(resume=True)

else:
    print("Starting new YOLO11m training from pretrained weights:", MODEL_NAME)

    model = YOLO(MODEL_NAME)
    model.add_callback("on_train_epoch_end", on_epoch_end_make_zip)

    results = model.train(
        data=str(aug_yaml_path),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,
        patience=PATIENCE,
        cache=CACHE,

        # Normal YOLO save location
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,

        # Save every epoch for resume safety
        save=True,
        save_period=SAVE_PERIOD,

        # Training stability/speed
        pretrained=True,
        optimizer="auto",
        amp=True,
        cos_lr=True,
        close_mosaic=YOLO_CLOSE_MOSAIC,
        seed=RANDOM_SEED,
        deterministic=False,
        plots=True,
        rect=False,      # keep false so mosaic/random perspective augmentations are fully active

        # Built-in YOLO augmentations optimized for tiny football detection.
        # Custom offline train-only augmentations already ran above.
        hsv_h=YOLO_HSV_H,
        hsv_s=YOLO_HSV_S,
        hsv_v=YOLO_HSV_V,
        translate=YOLO_TRANSLATE,
        scale=YOLO_SCALE,              # 0.50 -> random scale from 0.5x to 1.5x
        multi_scale=YOLO_MULTI_SCALE,  # random training image size, approx 0.5x to 1.5x
        fliplr=0.0,                    # avoid double horizontal flip because custom flip already happened
        mosaic=YOLO_MOSAIC,            # 4-image mosaic for small-object robustness
        mixup=YOLO_MIXUP,              # off so the small football does not disappear
        erasing=0.0,                   # off because random erasing can delete tiny footballs
    )

# Make/update the zip immediately after training completes normally.
zip_current_artifacts()

print("Run directory:")
print(RUN_DIR)
print("Best weights:")
print(RUN_DIR / "weights" / "best.pt")
print("Last checkpoint for resume:")
print(RUN_DIR / "weights" / "last.pt")
print("Download zip:")
print(ZIP_PATH)

display(FileLink(str(ZIP_PATH)))
if BEST_COPY_PATH.exists():
    display(FileLink(str(BEST_COPY_PATH)))
if LAST_COPY_PATH.exists():
    display(FileLink(str(LAST_COPY_PATH)))


In [ ]:
# =========================
# 11. VALIDATE BEST WEIGHTS
# =========================

from pathlib import Path
from ultralytics import YOLO

best_pt = RUN_DIR / "weights" / "best.pt"

if not best_pt.exists():
    found = list(Path("/kaggle/working").glob("**/best.pt"))
    if len(found) == 0:
        raise FileNotFoundError("No best.pt found yet. Train for at least 1 epoch first.")
    best_pt = max(found, key=lambda p: p.stat().st_mtime)

print("Validating:", best_pt)

val_model = YOLO(str(best_pt))
metrics = val_model.val(
    data=str(aug_yaml_path),
    imgsz=IMAGE_SIZE,
    device=0,
    plots=True
)

print(metrics)
print("Best weights saved at:")
print(best_pt)


## How to know it worked

Right before training, the notebook prints dataset counts.

For your earlier dataset, you probably want something close to:

```text
Train images: 2424
Valid images: 213
```

When YOLO starts, look for:

```text
train: Scanning ... 2424 images
val: Scanning ... 213 images
```

If train says `1212 images`, the augmentation step did not run before training.

### Resume behavior

If training gets paused after at least one epoch, rerun the notebook/cells. The training cell checks for:

```text
/kaggle/working/runs/football_detector_yolo11m_aug/weights/last.pt
```

and resumes automatically.

If the whole Kaggle session resets, upload/add your previously downloaded `MUST_DOWNLOAD_yolo11m_weights.zip` as a Kaggle input. The training cell will only resume from it if the zip metadata says it was created by this YOLO11m notebook.


In [ ]:
# =========================
# 12. EXPORT / DOWNLOAD ZIP ANYTIME
# =========================
# Run this cell whenever you want to refresh the downloadable weights zip.
# This works even if training was paused, as long as at least one epoch finished and last.pt exists.

from IPython.display import FileLink, display, Javascript
from pathlib import Path
import shutil
import zipfile

# Recreate zip_current_artifacts if this cell is run after a kernel restart.
if "zip_current_artifacts" not in globals():
    def zip_current_artifacts(zip_path=ZIP_PATH, quiet=False):
        weights_dir = RUN_DIR / "weights"
        best_pt = weights_dir / "best.pt"
        last_pt = weights_dir / "last.pt"

        if not best_pt.exists() and not last_pt.exists():
            raise FileNotFoundError("No best.pt or last.pt found yet. Finish at least 1 epoch first.")

        if best_pt.exists():
            shutil.copy2(best_pt, BEST_COPY_PATH)
        if last_pt.exists():
            shutil.copy2(last_pt, LAST_COPY_PATH)

        metadata_dir = Path("/kaggle/working/yolo11m_export_metadata")
        metadata_dir.mkdir(parents=True, exist_ok=True)
        (metadata_dir / "MODEL_NAME.txt").write_text(MODEL_NAME)
        (metadata_dir / "RUN_NAME.txt").write_text(RUN_NAME)
        (metadata_dir / "IMAGE_SIZE.txt").write_text(str(IMAGE_SIZE))
        (metadata_dir / "EPOCHS_TARGET.txt").write_text(str(EPOCHS))

        include_files = []
        include_files.extend(list(weights_dir.glob("*.pt")))

        for pattern in [
            "args.yaml",
            "results.csv",
            "results.png",
            "confusion_matrix.png",
            "confusion_matrix_normalized.png",
            "F1_curve.png",
            "P_curve.png",
            "PR_curve.png",
            "R_curve.png",
            "labels.jpg",
            "train_batch*.jpg",
            "val_batch*.jpg",
        ]:
            include_files.extend(list(RUN_DIR.glob(pattern)))

        if "aug_yaml_path" in globals() and Path(aug_yaml_path).exists():
            include_files.append(Path(aug_yaml_path))

        include_files.extend(list(metadata_dir.glob("*.txt")))

        seen = set()
        unique_files = []
        for p in include_files:
            p = Path(p)
            if p.exists() and p not in seen:
                unique_files.append(p)
                seen.add(p)

        with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for p in unique_files:
                if p.is_relative_to(Path("/kaggle/working")):
                    arcname = p.relative_to(Path("/kaggle/working"))
                else:
                    arcname = p.name
                zf.write(p, arcname=str(arcname))

        if not quiet:
            print(f"Created/updated: {zip_path}")
            print(f"Zip size: {zip_path.stat().st_size / (1024*1024):.1f} MB")

        return zip_path

zip_file = zip_current_artifacts()

print("Download links:")
display(FileLink(str(zip_file)))

if BEST_COPY_PATH.exists():
    display(FileLink(str(BEST_COPY_PATH)))

if LAST_COPY_PATH.exists():
    display(FileLink(str(LAST_COPY_PATH)))

# Browser auto-click is not guaranteed on Kaggle, but this sometimes opens the download prompt.
display(Javascript(f'''
(() => {{
  const link = document.querySelector('a[href$="{ZIP_PATH.name}"]');
  if (link) link.click();
}})();
'''))
